## Installing initial dependencies

In [2]:
!pip install gdown 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [gdown]


In [ ]:
import gdown

file_id = "11hRKwkFA46mD5W2OCoM-FvXTsn-73Zw6"  # no trailing slash
output_file = "individual_files_testing_segment.tfrecord"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output_file, quiet=False)

In [4]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip uninstall tensorflow -y
!{sys.executable} -m pip install tensorflow==2.12.0
!{sys.executable} -m pip install waymo-open-dataset-tf-2-12-0

Found existing installation: tensorflow 2.12.0
Uninstalling tensorflow-2.12.0:
  Successfully uninstalled tensorflow-2.12.0
  Using cached tensorflow-2.12.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached jax-0.4.30-py3-none-any.whl.metadata (22 kB)
  Using cached jaxlib-0.4.30-cp39-cp39-manylinux2014_x86_64.whl.metadata (1.0 kB)
  Using cached ml_dtypes-0.5.4-cp39-cp39-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached tensorflow-2.12.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (585.9 MB)
Using cached jax-0.4.30-py3-none-any.whl (2.0 MB)
Using cached jaxlib-0.4.30-cp39-cp39-manylinux2014_x86_64.whl (79.6 MB)
Using cached ml_dtypes-0.5.4-cp39-cp39-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tensorflow]4 [tensorflow]


In [5]:
import tensorflow as tf
print("TF version:", tf.__version__)

from waymo_open_dataset import dataset_pb2 as open_dataset
from waymo_open_dataset.utils import frame_utils

print("Waymo Open Dataset working.")

# Installing cupy for a gpu based numpy hadling -- Comment out this line if we are not using a cuda graphic card
# !pip install cupy --pre

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Creating audit report for all the t_records present in the folder

In [8]:
import os
import time
# import cupy as cp
import numpy as np
import csv
# import tensorflow.compat.v1 as tf
import tensorflow as tf
# tf.enable_eager_execution()
from PIL import Image

# tf.enable_eager_execution()

from waymo_open_dataset.utils import frame_utils
from waymo_open_dataset import dataset_pb2 as open_dataset

# CSV CONFIG
DATASET_DIR = "/home/jiale/Documents/eco_cars/data"  
CSV_OUTPUT_DIR = "/home/jiale/Documents/eco_cars/audit"

# Thresholds
DENSITY_MIN = 0.6
DENSITY_MAX = 2
QUALIFY_THRESHOLD = 0.9

# Creating the audit folder if it was not already existing
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

from typing import Optional

def compute_frame_density(frame) -> Optional[float]:
    # Decode FRONT camera
    front_image = None
    for img in frame.images:
        if img.name == open_dataset.CameraName.FRONT:
            front_image = tf.image.decode_jpeg(img.image).numpy()
            break
    if front_image is None:
        return None

    height, width = front_image.shape[:2]
    total_pixels = height * width

    # Parse range images — exactly as original
    (range_images,
        camera_projections,
        _,
        range_image_top_pose) = frame_utils.parse_range_image_and_camera_projection(frame)

    points_ri1, cp_ri1 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=0)
    points_ri2, cp_ri2 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=1)

    points_all = np.concatenate(points_ri1 + points_ri2, axis=0)
    cp_all     = np.concatenate(cp_ri1    + cp_ri2,    axis=0)

    # Get extrinsic — exactly as original
    extrinsic = None
    for calib in frame.context.camera_calibrations:
        if calib.name == open_dataset.CameraName.FRONT:
            extrinsic = np.array(calib.extrinsic.transform).reshape(4, 4)
            break
    if extrinsic is None:
        return None

    # Transform ALL points to camera space — exactly as original
    points_h = np.concatenate(
        [points_all, np.ones((points_all.shape[0], 1))],  # float64, no cast
        axis=1
    )
    vehicle_to_camera = np.linalg.inv(extrinsic)           # float64, no cast
    points_cam = (vehicle_to_camera @ points_h.T).T        # exactly original formula
    depths = points_cam[:, 2]

    # Filter FRONT camera — exactly as original
    front_mask = cp_all[:, 0] == open_dataset.CameraName.FRONT
    u = cp_all[front_mask][:, 1].astype(np.int32)
    v = cp_all[front_mask][:, 2].astype(np.int32)
    z = depths[front_mask]

    # Valid mask — exactly as original
    valid = (
        (u >= 0) & (u < width) &
        (v >= 0) & (v < height) &
        (z > 0)
    )
    u = u[valid]
    v = v[valid]
    z = z[valid]

    # Z-buffer loop — exactly as original
    depth_map = np.full((height, width), np.inf, dtype=np.float32)
    for ui, vi, zi in zip(u, v, z):
        if zi < depth_map[vi, ui]:
            depth_map[vi, ui] = zi
    depth_map[depth_map == np.inf] = np.nan

    # Density
    # valid_pixel_count = np.sum(~np.isnan(depth_map))
    # density_pct = (valid_pixel_count / total_pixels) * 100
    valid_mask = ~np.isnan(depth_map) & (depth_map > 0)
    density_pct = 100 * np.sum(valid_mask) / depth_map.size
    return float(density_pct)

# Function to create the dict array 
def audit_tfrecord(tfrecord_path: str) -> dict:
    segment_name = os.path.splitext(os.path.basename(tfrecord_path))[0]
    dataset = tf.data.TFRecordDataset(tfrecord_path, compression_type='')

    densities = []
    frame_idx = 0
    t0 = time.time()

    for raw_data in dataset:
        frame = open_dataset.Frame()
        frame.ParseFromString((raw_data.numpy()))
        density = compute_frame_density(frame)

        if density is not None:
            densities.append(density)
            if True:
                band = "✓" if DENSITY_MIN <= density <= DENSITY_MAX else "✗"
                print(f"    frame {frame_idx:04d} | density={density:.3f}% {band}")
        frame_idx += 1

    elapsed = time.time() - t0

    if not densities:
        return {
            "segment": segment_name, "total_frames": frame_idx,
            "sampled_frames": 0, "good_frames": 0, "good_pct": 0.0,
            "mean_density": 0.0, "min_density": 0.0, "max_density": 0.0,
            "median_density": 0.0, "label": "NOT SUITABLE",
            "reason": "no valid frames extracted", "elapsed_s": round(elapsed,1)
        }

    densities_arr = np.array(densities)
    good_mask = (densities_arr >= DENSITY_MIN) & (densities_arr <= DENSITY_MAX)
    good_frames = int(np.sum(good_mask))
    good_pct = good_frames / len(densities)

    if good_pct >= QUALIFY_THRESHOLD:
        label = "SUITABLE"
        reason = f"{good_pct*100:.1f}% frames within [{DENSITY_MIN}%, {DENSITY_MAX}%]"
    else:
        dead = int(np.sum(densities_arr < DENSITY_MIN))
        too_high = int(np.sum(densities_arr > DENSITY_MAX))
        label = "NOT SUITABLE"
        reason = f"{good_pct*100:.1f}% qualify (dead<{DENSITY_MIN}%: {dead}, above {DENSITY_MAX}%: {too_high})"

    return {
        "segment": segment_name, "total_frames": frame_idx,
        "sampled_frames": len(densities), "good_frames": good_frames,
        "good_pct": round(good_pct*100,2),
        "mean_density": round(float(np.mean(densities_arr)),4),
        "min_density": round(float(np.min(densities_arr)),4),
        "max_density": round(float(np.max(densities_arr)),4),
        "median_density": round(float(np.median(densities_arr)),4),
        "label": label, "reason": reason, "elapsed_s": round(elapsed,1)
    }

# RUN AUDIT
results = []
tfrecord_files = sorted(
    os.path.join(DATASET_DIR, f) for f in os.listdir(DATASET_DIR) if f.endswith(".tfrecord")
)

for i, tf_path in enumerate(tfrecord_files):
    print(f"[{i+1}/{len(tfrecord_files)}] {os.path.basename(tf_path)}")
    row = audit_tfrecord(tf_path)
    results.append(row)
    print(f"  → {row['label']} | {row['reason']} | mean={row['mean_density']:.3f}% ({row['elapsed_s']}s)\n")

# Write CSV
csv_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_results.csv")
fieldnames = ["segment","label","good_pct","mean_density","min_density","max_density",
              "median_density","good_frames","sampled_frames","total_frames","reason","elapsed_s"]
with open(csv_path,"w",newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

# Write summary
summary_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_summary.txt")
with open(summary_path,"w") as f:
    f.write("WAYMO DENSITY AUDIT SUMMARY\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Density band: [{DENSITY_MIN}%, {DENSITY_MAX}%] | threshold={QUALIFY_THRESHOLD*100:.0f}%\n")
    f.write("─"*60+"\n\n")
    suitable = [r for r in results if r["label"]=="SUITABLE"]
    not_suitable = [r for r in results if r["label"]!="SUITABLE"]
    f.write(f"SUITABLE ({len(suitable)}/{len(results)})\n")
    for r in suitable:
        f.write(f"  {r['segment']}\n")
        f.write(f"    good={r['good_pct']:.1f}% mean={r['mean_density']:.3f}% "
                f"[{r['min_density']:.3f}%, {r['max_density']:.3f}%]\n")
    f.write(f"\nNOT SUITABLE ({len(not_suitable)}/{len(results)})\n")
    for r in not_suitable:
        f.write(f"  {r['segment']}\n    {r['reason']}\n")

print(f"DONE. CSV: {csv_path} | Summary: {summary_path}")


[1/1] individual_files_testing_segment-10084636266401282188_1120_000_1140_000_with_camera_labels(1).tfrecord


2026-04-08 14:29:00.981324: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


    frame 0000 | density=0.199% ✗
    frame 0001 | density=0.197% ✗
    frame 0002 | density=0.195% ✗
    frame 0003 | density=0.194% ✗
    frame 0004 | density=0.199% ✗
    frame 0005 | density=0.187% ✗
    frame 0006 | density=0.192% ✗
    frame 0007 | density=0.196% ✗
    frame 0008 | density=0.200% ✗
    frame 0009 | density=0.188% ✗
    frame 0010 | density=0.190% ✗
    frame 0011 | density=0.201% ✗
    frame 0012 | density=0.202% ✗
    frame 0013 | density=0.197% ✗
    frame 0014 | density=0.197% ✗
    frame 0015 | density=0.185% ✗
    frame 0016 | density=0.198% ✗
    frame 0017 | density=0.199% ✗
    frame 0018 | density=0.194% ✗
    frame 0019 | density=0.191% ✗
    frame 0020 | density=0.203% ✗
    frame 0021 | density=0.200% ✗
    frame 0022 | density=0.198% ✗
    frame 0023 | density=0.195% ✗
    frame 0024 | density=0.196% ✗
    frame 0025 | density=0.194% ✗
    frame 0026 | density=0.197% ✗
    frame 0027 | density=0.199% ✗
    frame 0028 | density=0.195% ✗
    frame 0029

In [9]:
import os
import time
import tensorflow.compat.v1 as tf
import numpy as np
import csv
from PIL import Image

tf.enable_eager_execution()

from waymo_open_dataset.utils import frame_utils
from waymo_open_dataset import dataset_pb2 as open_dataset

# CSV CONFIG
DATASET_DIR = "/home/jiale/Documents/eco_cars/data"
CSV_OUTPUT_DIR = "/home/jiale/Documents/eco_cars/audit"

# Thresholds
DENSITY_MIN = 0.6
DENSITY_MAX = 2
QUALIFY_THRESHOLD = 0.9

os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

from typing import Optional

def compute_frame_density(frame) -> Optional[float]:
    # Extract FRONT Camera Image — identical to original
    front_image = None
    for img in frame.images:
        if img.name == open_dataset.CameraName.FRONT:
            front_image = tf.image.decode_jpeg(img.image).numpy()
            break

    if front_image is None:
        return None

    height, width = front_image.shape[:2]

    # Parse Range Images — identical to original
    (range_images,
     camera_projections,
     _,
     range_image_top_pose) = frame_utils.parse_range_image_and_camera_projection(frame)

    # Convert BOTH returns to point cloud — identical to original
    points_ri1, cp_ri1 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=0)
    points_ri2, cp_ri2 = frame_utils.convert_range_image_to_point_cloud(
        frame, range_images, camera_projections, range_image_top_pose, ri_index=1)

    # Concatenate all lidars — identical to original
    points_all = np.concatenate(points_ri1 + points_ri2, axis=0)
    cp_all = np.concatenate(cp_ri1 + cp_ri2, axis=0)

    # Get extrinsic — identical to original
    extrinsic = None
    for calib in frame.context.camera_calibrations:
        if calib.name == open_dataset.CameraName.FRONT:
            extrinsic = np.array(calib.extrinsic.transform).reshape(4, 4)
            break

    if extrinsic is None:
        return None

    # Convert vehicle -> camera frame — identical to original
    points_h = np.concatenate(
        [points_all, np.ones((points_all.shape[0], 1))],
        axis=1
    )

    vehicle_to_camera = np.linalg.inv(extrinsic)
    points_cam = (vehicle_to_camera @ points_h.T).T

    # True optical depth — identical to original
    depths = points_cam[:, 2]

    # Filter FRONT camera projections — identical to original
    front_mask = cp_all[:, 0] == open_dataset.CameraName.FRONT

    u = cp_all[front_mask][:, 1].astype(np.int32)
    v = cp_all[front_mask][:, 2].astype(np.int32)
    z = depths[front_mask]

    # Keep valid pixels — identical to original
    valid = (
        (u >= 0) & (u < width) &
        (v >= 0) & (v < height) &
        (z > 0)
    )

    u = u[valid]
    v = v[valid]
    z = z[valid]

    # Create Sparse Depth Map (Z-buffer) — identical to original
    depth_map = np.full((height, width), np.inf, dtype=np.float32)

    for ui, vi, zi in zip(u, v, z):
        if zi < depth_map[vi, ui]:
            depth_map[vi, ui] = zi

    depth_map[depth_map == np.inf] = np.nan

    # Density — identical to original viz script
    valid_mask = ~np.isnan(depth_map) & (depth_map > 0)
    density_pct = 100 * np.sum(valid_mask) / depth_map.size

    return float(density_pct)


def audit_tfrecord(tfrecord_path: str) -> dict:
    segment_name = os.path.splitext(os.path.basename(tfrecord_path))[0]
    dataset = tf.data.TFRecordDataset(tfrecord_path, compression_type='')

    densities = []
    frame_idx = 0
    t0 = time.time()

    for raw_data in dataset:
        frame = open_dataset.Frame()
        frame.ParseFromString(bytearray(raw_data.numpy()))
        density = compute_frame_density(frame)

        if density is not None:
            densities.append(density)
            band = "✓" if DENSITY_MIN <= density <= DENSITY_MAX else "✗"
            print(f"  frame {frame_idx:04d} | density={density:.4f}% {band}")

        frame_idx += 1

    elapsed = time.time() - t0

    if not densities:
        return {
            "segment": segment_name, "total_frames": frame_idx,
            "sampled_frames": 0, "good_frames": 0, "good_pct": 0.0,
            "mean_density": 0.0, "min_density": 0.0, "max_density": 0.0,
            "median_density": 0.0, "label": "NOT SUITABLE",
            "reason": "no valid frames extracted", "elapsed_s": round(elapsed, 1)
        }

    densities_arr = np.array(densities)
    good_mask = (densities_arr >= DENSITY_MIN) & (densities_arr <= DENSITY_MAX)
    good_frames = int(np.sum(good_mask))
    good_pct = good_frames / len(densities)

    if good_pct >= QUALIFY_THRESHOLD:
        label = "SUITABLE"
        reason = f"{good_pct*100:.1f}% frames within [{DENSITY_MIN}%, {DENSITY_MAX}%]"
    else:
        dead = int(np.sum(densities_arr < DENSITY_MIN))
        too_high = int(np.sum(densities_arr > DENSITY_MAX))
        label = "NOT SUITABLE"
        reason = f"{good_pct*100:.1f}% qualify (dead<{DENSITY_MIN}%: {dead}, above {DENSITY_MAX}%: {too_high})"

    return {
        "segment": segment_name, "total_frames": frame_idx,
        "sampled_frames": len(densities), "good_frames": good_frames,
        "good_pct": round(good_pct * 100, 2),
        "mean_density": round(float(np.mean(densities_arr)), 4),
        "min_density": round(float(np.min(densities_arr)), 4),
        "max_density": round(float(np.max(densities_arr)), 4),
        "median_density": round(float(np.median(densities_arr)), 4),
        "label": label, "reason": reason, "elapsed_s": round(elapsed, 1)
    }


# RUN AUDIT
results = []
tfrecord_files = sorted(
    os.path.join(DATASET_DIR, f) for f in os.listdir(DATASET_DIR) if f.endswith(".tfrecord")
)

for i, tf_path in enumerate(tfrecord_files):
    print(f"[{i+1}/{len(tfrecord_files)}] {os.path.basename(tf_path)}")
    row = audit_tfrecord(tf_path)
    results.append(row)
    print(f"  → {row['label']} | {row['reason']} | mean={row['mean_density']:.4f}% ({row['elapsed_s']}s)\n")

# Write CSV
csv_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_results.csv")
fieldnames = ["segment", "label", "good_pct", "mean_density", "min_density", "max_density",
              "median_density", "good_frames", "sampled_frames", "total_frames", "reason", "elapsed_s"]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

# Write summary
summary_path = os.path.join(CSV_OUTPUT_DIR, "waymo_audit_summary.txt")
with open(summary_path, "w") as f:
    f.write("WAYMO DENSITY AUDIT SUMMARY\n")
    f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Density band: [{DENSITY_MIN}%, {DENSITY_MAX}%] | threshold={QUALIFY_THRESHOLD*100:.0f}%\n")
    f.write("─" * 60 + "\n\n")
    suitable = [r for r in results if r["label"] == "SUITABLE"]
    not_suitable = [r for r in results if r["label"] != "SUITABLE"]
    f.write(f"SUITABLE ({len(suitable)}/{len(results)})\n")
    for r in suitable:
        f.write(f"  {r['segment']}\n")
        f.write(f"  good={r['good_pct']:.1f}% mean={r['mean_density']:.4f}% "
                f"[{r['min_density']:.4f}%, {r['max_density']:.4f}%]\n")
    f.write(f"\nNOT SUITABLE ({len(not_suitable)}/{len(results)})\n")
    for r in not_suitable:
        f.write(f"  {r['segment']}\n  {r['reason']}\n")

print(f"DONE. CSV: {csv_path} | Summary: {summary_path}")

[1/1] individual_files_testing_segment-10084636266401282188_1120_000_1140_000_with_camera_labels(1).tfrecord


2026-04-08 14:33:38.861908: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


  frame 0000 | density=0.1986% ✗
  frame 0001 | density=0.1974% ✗
  frame 0002 | density=0.1954% ✗
  frame 0003 | density=0.1938% ✗
  frame 0004 | density=0.1991% ✗
  frame 0005 | density=0.1872% ✗
  frame 0006 | density=0.1920% ✗
  frame 0007 | density=0.1964% ✗
  frame 0008 | density=0.1995% ✗
  frame 0009 | density=0.1882% ✗
  frame 0010 | density=0.1903% ✗
  frame 0011 | density=0.2011% ✗
  frame 0012 | density=0.2018% ✗
  frame 0013 | density=0.1966% ✗
  frame 0014 | density=0.1975% ✗
  frame 0015 | density=0.1850% ✗
  frame 0016 | density=0.1978% ✗
  frame 0017 | density=0.1993% ✗
  frame 0018 | density=0.1939% ✗
  frame 0019 | density=0.1913% ✗
  frame 0020 | density=0.2034% ✗
  frame 0021 | density=0.2001% ✗
  frame 0022 | density=0.1976% ✗
  frame 0023 | density=0.1947% ✗
  frame 0024 | density=0.1964% ✗
  frame 0025 | density=0.1944% ✗
  frame 0026 | density=0.1965% ✗
  frame 0027 | density=0.1989% ✗
  frame 0028 | density=0.1953% ✗
  frame 0029 | density=0.1993% ✗
  frame 00